# SIG30 sPCA signature scoring (SIG18 programs)

Projects the SIG18 sparse-PCA programs onto the SIG30 bulk samples (single
CRISPR-KO target vs. control, one dataset, no timepoints), using decoupler
weighted `waggr` on z-scored (scaled) expression, with the **top-50 weighted**
net per component (loadings as weights; matches cytosig's net_50_weighted),
mirroring the CytoSig first step.

SIG18 program genes and SIG30 are **both mouse**, so (unlike the SIG26 port)
no mouse->human gene conversion is needed here.

# Imports

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import decoupler as dc

import sys
import os
from pathlib import Path

# Repo-relative paths: inputs from imports_stable/, outputs to analysis_outs/<module>/
_here = Path.cwd().resolve()
REPO_DIR = str(next(p for p in [_here, *_here.parents] if (p / "imports_stable").is_dir()))
IMPORTS_DIR = f"{REPO_DIR}/imports_stable"
OUTS_DIR = f"{REPO_DIR}/analysis_outs/07_crispr_arrayed_validation_RNAseq_SIG30"
sys.path.insert(0, f"{REPO_DIR}/functions")
import scanpy_custom as scc

import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

# Config: program source and paths

In [ ]:
SIG30_PROC = f"{IMPORTS_DIR}/SIG30/processing_outs"
OUT = f"{OUTS_DIR}/inference_SIG18"
os.makedirs(OUT, exist_ok=True)
TOP_N = 50          # genes per program used to build the waggr net

# Wide gene x component loadings, and the lm file giving clean comps
SOURCES = {
    "SIG18": dict(
        components=IMPORTS_DIR + "/SIG18/analysis_outs/spca/zscore_degs/zscore_degs_unscaled_alpha10.0_sPCA_components.csv",
        clean=IMPORTS_DIR + "/SIG18/analysis_outs/spca/lm_fit_zscore_degs_alpha10.0_sPCA_clean.csv",
    ),
}

# Load programs

For SIG18: restrict to clean components, melt to long
(gene, component, loading), and build a weighted net (source/target/weight)
for `waggr` from the top-50 genes per component (by loading).
No mouse->human gene conversion (both SIG18 and SIG30 are mouse).

In [ ]:
def load_programs(name):
    cfg = SOURCES[name]
    feats = pd.read_csv(cfg["components"])                 # cols: gene, 0, 1, 2, ...
    clean = pd.read_csv(cfg["clean"])
    clean_comps = [c.replace("comp_", "") for c in clean["component"].unique()]
    clean_comps = [c for c in clean_comps if c in feats.columns]  # keep available

    # long format over clean components only
    long = (feats.melt(id_vars="gene", value_vars=clean_comps,
                       var_name="component", value_name="loading")
                 .query("loading != 0")
                 .drop_duplicates(["component", "gene"]))

    # top-N genes per component (by loading)
    top = (long.sort_values("loading", ascending=False)
               .groupby("component", group_keys=False).head(TOP_N))

    # top-50 WEIGHTED net for waggr (loadings as weights), matching cytosig net_50_weighted
    net = (top[["component", "gene", "loading"]]
           .rename(columns={"component": "source", "gene": "target", "loading": "weight"}))
    net["source"] = "comp_" + net["source"].astype(str)

    print(f"{name}: {len(clean_comps)} clean components, top-{TOP_N} genes/component, "
          f"{top['gene'].nunique()} unique genes")
    return net

PROGRAMS = {name: load_programs(name) for name in SOURCES}

# Build SIG30 AnnData

Counts → gene symbols → `normalize_total` + `log1p` (`log1p_norm`), then `scale`
(`log1p_norm_scaled`). obs carries the sample metadata (single dataset, no
per-timepoint split).

In [ ]:
def build_adata():
    counts = pd.read_csv(f"{SIG30_PROC}/count_matrix_umiDeDup_SIG30.csv", index_col=0)
    feature_names = pd.read_csv(f"{SIG30_PROC}/featureNames_SIG30.csv", index_col=0)
    meta = pd.read_csv(f"{SIG30_PROC}/processed_metadata_SIG30.csv")

    cf = counts.loc[counts.index.isin(feature_names.index), :]
    cf = cf.merge(feature_names, left_index=True, right_index=True).set_index("gene").drop(columns=["category"])

    rna = sc.AnnData(cf.T)
    rna.var_names_make_unique()
    sc.pp.normalize_total(rna)
    sc.pp.log1p(rna)
    rna.layers["log1p_norm"] = rna.X.copy()
    sc.pp.scale(rna)
    rna.layers["log1p_norm_scaled"] = rna.X.copy()

    obs = (meta.set_index("sample_ID")
               .loc[rna.obs_names, ["target", "replicate",
                                    "total_reads_counts", "seq_saturation"]])
    rna.obs = obs
    return rna

# Score programs (waggr, weighted z-scored expression)

In [ ]:
def waggr_method(rna):
    base = rna.copy()
    obs = base.obs.copy()
    for src, net in PROGRAMS.items():
        genes = net["target"].unique().tolist()
        sub = base[:, base.var_names.isin(genes)].copy()
        sub.X = sub.layers["log1p_norm"]
        sc.pp.scale(sub)                                  # z-score expression
        dc.mt.waggr(sub, net, tmin=5, times=0)           # weighted activity, no perms
        sdf = sub.obsm["score_waggr"].copy()
        sdf.columns = [f"{src}_waggr_{c}" for c in sdf.columns]
        obs = pd.concat([obs, sdf.set_index(sub.obs_names)], axis=1)
    waggr_cols = [c for c in obs.columns if "_waggr_" in c]
    obs.to_csv(f"{OUT}/waggr_scored_obs_SIG30.csv")
    print(f"[waggr] SIG30: {len(waggr_cols)} waggr columns -> waggr_scored_obs_SIG30.csv")
    return obs

# Run waggr scoring

In [ ]:
print("==== SIG30 ====")
rna = build_adata()
print("adata:", rna.shape)
wg = waggr_method(rna)

In [ ]:
# quick peek
print("waggr cols (first 4):", [c for c in wg.columns if "_waggr_" in c][:4])
wg.filter(like="_waggr_").iloc[:3, :3]